1. Load dataset & Keep final test set

In [2]:
import pandas as pd

from sklearn.model_selection import train_test_split
from preprocess import preprocess


# =========================
# 1. Load dataset
# =========================

df = pd.read_json("../donnees/articles.json")

X = df["body"]
y = df["categories"].str[0]


# =========================
# 2. Keep final test set
# =========================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


2. Development Data

In [3]:
print("=== DEVELOPMENT DATA ===")

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("\nNumber of samples:", len(X_train))

print("\nCategory distribution:")
print(y_train.value_counts())

=== DEVELOPMENT DATA ===
X_train shape: (16800,)
y_train shape: (16800,)

Number of samples: 16800

Category distribution:
categories
ثقافة     2800
دولي      2800
اقتصاد    2800
رياضة     2800
سياسة     2800
مجتمع     2800
Name: count, dtype: int64


3. Preprocessing

In [4]:
X_train = X_train.apply(preprocess)
X_test = X_test.apply(preprocess)

4. 5-fold cross-validation

In [5]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("Number of folds:", skf.get_n_splits())

Number of folds: 5


5. Verify the folds

In [6]:
for fold, (train_index, val_index) in enumerate(
    skf.split(X_train, y_train),
    start=1
):
    print(f"\n========== FOLD {fold} ==========")

    print("Training size:", len(train_index))
    print("Validation size:", len(val_index))

    print("\nValidation distribution:")
    print(
        y_train.iloc[val_index]
        .value_counts()
        .sort_index()
    )


========== FOLD 1 ==========
Training size: 13440
Validation size: 3360

Validation distribution:
categories
اقتصاد    560
ثقافة     560
دولي      560
رياضة     560
سياسة     560
مجتمع     560
Name: count, dtype: int64

========== FOLD 2 ==========
Training size: 13440
Validation size: 3360

Validation distribution:
categories
اقتصاد    560
ثقافة     560
دولي      560
رياضة     560
سياسة     560
مجتمع     560
Name: count, dtype: int64

========== FOLD 3 ==========
Training size: 13440
Validation size: 3360

Validation distribution:
categories
اقتصاد    560
ثقافة     560
دولي      560
رياضة     560
سياسة     560
مجتمع     560
Name: count, dtype: int64

========== FOLD 4 ==========
Training size: 13440
Validation size: 3360

Validation distribution:
categories
اقتصاد    560
ثقافة     560
دولي      560
رياضة     560
سياسة     560
مجتمع     560
Name: count, dtype: int64

========== FOLD 5 ==========
Training size: 13440
Validation size: 3360

Validation distribution:
categories
اقتصاد    

6. TF-IDF & Logistic Regression & Macro F1

In [7]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

def run_cv(ngram_range=(1, 1), min_df=2, C=1.0):

    fold_scores = []

    for fold, (train_index, val_index) in enumerate(
        skf.split(X_train, y_train),
        start=1
    ):

        X_fold_train = X_train.iloc[train_index]
        X_fold_val = X_train.iloc[val_index]

        y_fold_train = y_train.iloc[train_index]
        y_fold_val = y_train.iloc[val_index]

        # TF-IDF
        vectorizer = TfidfVectorizer(
            min_df=min_df,
            ngram_range=ngram_range
        )

        X_fold_train_tfidf = vectorizer.fit_transform(
            X_fold_train
        )

        X_fold_val_tfidf = vectorizer.transform(
            X_fold_val
        )

        # Logistic Regression
        lr_model = LogisticRegression(
            C=C,
            max_iter=10000
        )

        lr_model.fit(
            X_fold_train_tfidf,
            y_fold_train
        )

        # Prediction
        y_fold_pred = lr_model.predict(
            X_fold_val_tfidf
        )

        # Macro F1
        fold_f1 = f1_score(
            y_fold_val,
            y_fold_pred,
            average="macro"
        )

        fold_scores.append(fold_f1)

    return np.mean(fold_scores), np.std(fold_scores)

7. Mean/Std

In [8]:
baseline_mean, baseline_std = run_cv(
    ngram_range=(1, 1),
    min_df=2,
    C=1.0
)

print("Unigrams")
print("Mean Macro F1:", baseline_mean)
print("Std:", baseline_std)

Unigrams
Mean Macro F1: 0.8844296818814342
Std: 0.004727562190965272


In [9]:
import sys

print(sys.version)

try:
    import sentence_transformers
    print("sentence-transformers:", sentence_transformers.__version__)
except ImportError:
    print("sentence-transformers: NOT INSTALLED")

try:
    import torch
    print("torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
except ImportError:
    print("torch: NOT INSTALLED")

3.14.0 (tags/v3.14.0:ebf955d, Oct  7 2025, 10:15:03) [MSC v.1944 64 bit (AMD64)]
sentence-transformers: 6.0.1
torch: 2.13.0+cpu
CUDA available: False
